In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    StandardScaler
)

import joblib

In [3]:
dataset_path = "../data/raw/Churn_Modelling.csv"

df = pd.read_csv(dataset_path)

df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


# Dropping Unnecessary Features

The following features are removed because they do not contribute meaningful predictive information.

- RowNumber
- CustomerId
- Surname

In [4]:
columns_to_drop = ["RowNumber", "CustomerId", "Surname"]
df = df.drop(columns = columns_to_drop)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


# Feature and Target Separation

Separating the independent features (X) and the target variable (y).

In [6]:
X = df.drop('Exited', axis=1)
Y = df['Exited']

In [11]:
X.columns

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary'],
      dtype='str')

In [13]:
print("Feature Shape :", X.shape)
print("Target Shape  :", Y.shape)

Feature Shape : (10000, 10)
Target Shape  : (10000,)


In [14]:
Y.head()

0    1
1    0
2    1
3    0
4    0
Name: Exited, dtype: int64

## Observation

- The dataset has been successfully separated into features (X) and target (y).

- X contains 10 predictor variables.

- y contains the binary target variable "Exited".

- The data is now ready for train-test splitting.

In [15]:
X.shape

(10000, 10)

In [16]:
Y.shape

(10000,)

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

In [18]:
print("X_Train shape : ", X_train.shape)
print("X_test shape : ", X_test.shape)

print("Y_Train shape : ", Y_train.shape)
print("Y_test shape : ", Y_test.shape)

X_Train shape :  (8000, 10)
X_test shape :  (2000, 10)
Y_Train shape :  (8000,)
Y_test shape :  (2000,)


In [20]:
print("Training Set")
print((Y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting Set")
print((Y_test.value_counts(normalize=True) * 100).round(2))

Training Set
Exited
0    79.62
1    20.38
Name: proportion, dtype: float64

Testing Set
Exited
0    79.65
1    20.35
Name: proportion, dtype: float64


In [21]:
categorical_features = [
    "Geography",
    "Gender"
]

numerical_features = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [23]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(drop="first", handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [25]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [26]:
print("Processed X_train:", X_train_processed.shape)
print("Processed X_test :", X_test_processed.shape)

Processed X_train: (8000, 11)
Processed X_test : (2000, 11)


In [29]:
X.head()


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,France,Female,42,2,0.00,1,1,1,101348.88
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58
2,502,France,Female,42,8,159660.80,3,1,0,113931.57
3,699,France,Female,39,1,0.00,2,0,0,93826.63
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10


In [30]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_train_processed_df.head()

,num__CreditScore,num__Age,num__Tenure,num__Balance,num__NumOfProducts,num__HasCrCard,num__IsActiveMember,num__EstimatedSalary,cat__Geography_Germany,cat__Geography_Spain,cat__Gender_Male
2151,1.058568,1.715086,0.684723,-1.226059,-0.910256,0.641042,-1.030206,1.042084,0.0,0.0,1.0
8392,0.913626,-0.659935,-0.696202,0.413288,-0.910256,0.641042,-1.030206,-0.623556,1.0,0.0,1.0
5006,1.079274,-0.184931,-1.731895,0.601687,0.808830,0.641042,0.970680,0.308128,1.0,0.0,0.0
4117,-0.929207,-0.184931,-0.005739,-1.226059,0.808830,0.641042,-1.030206,-0.290199,0.0,0.0,1.0
7182,0.427035,0.955079,0.339492,0.548318,0.808830,-1.559960,0.970680,0.135042,1.0,0.0,1.0


In [31]:
feature_names = preprocessor.get_feature_names_out()

feature_names

array(['num__CreditScore', 'num__Age', 'num__Tenure', 'num__Balance',
       'num__NumOfProducts', 'num__HasCrCard', 'num__IsActiveMember',
       'num__EstimatedSalary', 'cat__Geography_Germany',
       'cat__Geography_Spain', 'cat__Gender_Male'], dtype=object)

In [32]:
import joblib

joblib.dump(preprocessor, "../models/preprocessor.pkl")

['../models/preprocessor.pkl']

In [33]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

In [34]:
X_train_processed_df.head()

,num__CreditScore,num__Age,num__Tenure,num__Balance,num__NumOfProducts,num__HasCrCard,num__IsActiveMember,num__EstimatedSalary,cat__Geography_Germany,cat__Geography_Spain,cat__Gender_Male
0,1.058568,1.715086,0.684723,-1.226059,-0.910256,0.641042,-1.030206,1.042084,0.0,0.0,1.0
1,0.913626,-0.659935,-0.696202,0.413288,-0.910256,0.641042,-1.030206,-0.623556,1.0,0.0,1.0
2,1.079274,-0.184931,-1.731895,0.601687,0.808830,0.641042,0.970680,0.308128,1.0,0.0,0.0
3,-0.929207,-0.184931,-0.005739,-1.226059,0.808830,0.641042,-1.030206,-0.290199,0.0,0.0,1.0
4,0.427035,0.955079,0.339492,0.548318,0.808830,-1.559960,0.970680,0.135042,1.0,0.0,1.0


In [35]:
X_train_processed_df.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_test_processed_df.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

In [39]:
Y_train.to_frame().to_csv(
    "../data/processed/y_train.csv",
    index=False
)

Y_test.to_frame().to_csv(
    "../data/processed/y_test.csv",
    index=False
)

In [37]:
import os

print(os.listdir("../data/processed"))

['X_test.csv', 'X_train.csv', 'Y_test.csv', 'Y_train.csv']


In [38]:
print(os.listdir("../models"))

['preprocessor.pkl']




## Completed Tasks

- Removed unnecessary columns
- Split features and target
- Performed train-test split
- Applied feature scaling
- Applied one-hot encoding
- Prevented data leakage by fitting preprocessing only on training data
- Saved the fitted preprocessor
- Saved processed training and testing datasets

The dataset is now fully prepared for machine learning model training.